# NFCorpus: 512 vs. 256 tokens (#3)

Controlled retrieval comparison on NFCorpus (**323 test queries**). Only `max_seq_length` changes; both runs use the pilot MiniLM, `title + text`, normalized embeddings, exact FAISS `IndexFlatIP`, and graded nDCG@10. One vector per document; **no chunking**.

SciFact and FiQA remain held out. These settings are **not** the final baseline.

Run from the repository root:: `experiments/baseline/nfcorpus_retrieval.py`

The notebook analyzes saved run outputs

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import spearmanr, wilcoxon
from transformers import AutoTokenizer
from transformers.utils import logging as hf_logging

from silent_rag.eval.ground_truth import bootstrap_ci, paired_bootstrap

hf_logging.set_verbosity_error()

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "experiments"))
from nfcorpus_retrieval import load_nfcorpus  # noqa: E402

RESULTS = ROOT / "results" / "baseline" / "nfcorpus"
TABLES = ROOT / "results" / "tables"
FIGURES = ROOT / "results" / "figures"
A, B = "minilm_trunc512", "minilm_trunc256"

corpus, query_ids, _, qrels = load_nfcorpus()
if any(not (RESULTS / name / "run.json").exists() or not (RESULTS / name / "per_query.csv").exists() for name in (A, B)):
    raise FileNotFoundError(
        "Missing NFCorpus retrieval runs. First run from the repository root: "
        "python experiments/nfcorpus_retrieval.py"
    )

runs = {name: json.loads((RESULTS / name / "run.json").read_text()) for name in (A, B)}
per_query = {name: pd.read_csv(RESULTS / name / "per_query.csv", dtype={"query_id": str}) for name in (A, B)}

## Check that only one setting differs

In [ ]:
diff_keys = {k: (runs[A]["config"][k], runs[B]["config"][k]) for k in runs[A]["config"] if runs[A]["config"][k] != runs[B]["config"][k]}
print("settings that differ:", diff_keys)
assert set(diff_keys) == {"max_seq_length"}
for key in ("corpus", "qrels", "versions", "torch_threads"):
    assert runs[A][key] == runs[B][key], key
assert per_query[A]["query_id"].tolist() == per_query[B]["query_id"].tolist()

cols = ["n_docs", "n_vectors", "n_queries", "query_tokens_max", "doc_inputs_over_max_seq_length", "doc_encode_seconds", "search_seconds", "ndcg@10", "ndcg@10_ci95"]
pd.DataFrame({name: {c: runs[name][c] for c in cols} for name in (A, B)})

`query_tokens_max` is far below 256, so the 256 setting only truncates documents. The encoding times are single threaded CPU and recorded only for comparing the two runs.

## Paired comparison

For each query: `diff = nDCG@10(512) − nDCG@10(256)`. Report the mean and 95% percentile bootstrap CI (10,000 resamples, seed 0), plus Wilcoxon and paired-bootstrap checks

In [ ]:
a = per_query[A]["ndcg@10"].to_numpy()
b = per_query[B]["ndcg@10"].to_numpy()
diff = a - b

mean_a, lo_a, hi_a = bootstrap_ci(a)
mean_b, lo_b, hi_b = bootstrap_ci(b)
mean_d, lo_d, hi_d = bootstrap_ci(diff)
drop, p_one_sided = paired_bootstrap(a, b)
w = wilcoxon(a, b)

paired = pd.DataFrame(
    [
        {"quantity": "nDCG@10 at 512", "mean": mean_a, "ci_lo": lo_a, "ci_hi": hi_a},
        {"quantity": "nDCG@10 at 256", "mean": mean_b, "ci_lo": lo_b, "ci_hi": hi_b},
        {"quantity": "paired diff (512 - 256)", "mean": mean_d, "ci_lo": lo_d, "ci_hi": hi_d},
    ]
).round(4)
paired.to_csv(TABLES / "nfcorpus_truncation_paired.csv", index=False)

print(f"512 better: {(diff > 0).sum()} | tied: {(diff == 0).sum()} | 256 better: {(diff < 0).sum()}")
print(f"wilcoxon signed-rank (two-sided, zeros dropped): p = {w.pvalue:.3f}")
print(f"team paired_bootstrap (one-sided, 2000 resamples): drop = {drop:.4f}, p = {p_one_sided:.3f}")
paired

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].scatter(b, a, s=10, alpha=0.6)
axes[0].plot([0, 1], [0, 1], color="grey", linewidth=0.8)
axes[0].set_xlabel("nDCG@10, max_seq_length 256")
axes[0].set_ylabel("nDCG@10, max_seq_length 512")
axes[0].set_title("Per query")
axes[1].hist(diff[diff != 0], bins=40)
axes[1].axvline(0, color="grey", linewidth=0.8)
axes[1].set_xlabel("per-query diff (512 - 256), ties left out")
axes[1].set_ylabel("queries")
axes[1].set_title(f"{(diff != 0).sum()} of {len(diff)} queries changed")
fig.tight_layout()
fig.savefig(FIGURES / "nfcorpus_truncation_paired.png", dpi=150)
plt.show()

**Sensitivity check:** Recompute the paired CI without queries with absolute differences over 0.5. They remain part of the main analysis.

In [ ]:
big = np.abs(diff) > 0.5
print("queries with |diff| > 0.5:", per_query[A].loc[big, "query_id"].tolist(), np.round(diff[big], 3))
print("relevant docs for those queries:", [len([d for d, r in qrels[q].items() if r > 0]) for q in per_query[A].loc[big, "query_id"]])
print("median per-query diff:", np.median(diff))
print("paired diff without them: mean %.4f, 95%% CI [%.4f, %.4f]" % bootstrap_ci(diff[~big]))

## Diagnostics

**How much does the top-10 change?** This is label-free: the overlap between the two runs' top-10 lists for each query

**Does truncation hurt more when the relevant documents are long?** This uses the NFCorpus dev qrels. Queries are grouped by the share of their relevant documents that are over 256 tokens, the documents that get cut at 256

In [ ]:
overlap = np.array(
    [len(set(x.split()) & set(y.split())) for x, y in zip(per_query[A]["top10"], per_query[B]["top10"])]
)
print("top-10 docs shared between the runs: mean", overlap.mean().round(2), "| identical top-10 sets:", (overlap == 10).sum(), "/", len(overlap))

assert query_ids == per_query[A]["query_id"].tolist()
tokenizer = AutoTokenizer.from_pretrained(runs[A]["config"]["model"], revision=runs[A]["config"]["model_revision"])
doc_text = (corpus["title"].fillna("").str.strip() + " " + corpus["text"].fillna("").str.strip()).str.strip()
doc_tokens = dict(zip(corpus["_id"], [len(ids) for ids in tokenizer(doc_text.tolist())["input_ids"]]))

long_share = np.array([np.mean([doc_tokens[d] > 256 for d, r in qrels[q].items() if r > 0]) for q in query_ids])
groups = pd.cut(long_share, [-0.01, 0.5, 0.8, 1.0], labels=["<= 50%", "50-80%", "> 80%"])
by_group = (
    pd.DataFrame({"share of relevant docs over 256 tokens": groups, "diff": diff})
    .groupby("share of relevant docs over 256 tokens", observed=True)["diff"]
    .agg(queries="size", mean_diff="mean", changed=lambda s: int((s != 0).sum()))
    .round(4)
)
rho = spearmanr(long_share, diff)
print(f"spearman (share of long relevant docs vs diff): rho = {rho.statistic:.3f}, p = {rho.pvalue:.3f}")
print(f"> 80% group without the |diff| > 0.5 queries: mean diff {diff[(long_share > 0.8) & ~big].mean():.4f}")
by_group

## Summary

| | nDCG@10 | 95% CI |
|---|---:|---|
| 512 tokens | 0.2967 | 0.2635–0.3302 |
| 256 tokens | 0.2917 | 0.2591–0.3251 |
| Paired difference (512 − 256) | +0.0050 | −0.0032 to +0.0143 |

- **No statistically clear advantage for 512.** Paired CI includes 0; Wilcoxon p = 0.39. This does **not** prove equivalence.
- **Three queries drive the mean gain.** Without them: −0.0006 (CI −0.0067 to +0.0053).
- **Document length is not a clear explanation.** Correlation with share of long relevant docs is weak (Spearman ρ = 0.078, p = 0.16); the apparent subgroup gain disappears without those three queries.
- **Rankings change:** Only 21/323 queries retain the same top-10 set; mean overlap 7.9/10.
- **CPU encoding time:** 71 s at 256 vs. 173 s at 512 (same machine).

One model and one dataset; SciFact/FiQA configuration comparisons remain held out. Chunking is outside this notebook.